In [ ]:
# ============================================================
# EVENT-BASED SYNC (UC Delta -> PostgreSQL RDS)
# - Incremental via Delta Change Data Feed (CDF) when available
# - FIRST RUN (no state): FULL SNAPSHOT backfill (spark.table)
# - Robust:
#   * Missing UC tables are skipped
#   * If CDF is missing for a version range, falls back to SNAPSHOT
# - Designed for Databricks Workflows / Table Trigger
# ============================================================

from pyspark.sql import functions as F
import psycopg2
from psycopg2.extras import execute_batch
import pandas as pd
import uuid, json, os
import boto3
import botocore

import time, traceback
from datetime import datetime, timezone


# ============================================================
# 0) LOGGER
# ============================================================
def log(event, **kwargs):
    ts = datetime.now(timezone.utc).isoformat()
    kv = " ".join([f"{k}={repr(v)}" for k, v in kwargs.items()])
    print(f"[{ts}] {event} {kv}")


# ============================================================
# CREDENTIALS RETRIEVAL (AWS SM)
# ============================================================
SERVICE_CREDENTIAL_NAME = "asset_api_nprd_rds_service_credential"
REGION = "eu-west-1"

provider = dbutils.credentials.getServiceCredentialsProvider(SERVICE_CREDENTIAL_NAME)

session = boto3.Session(
    botocore_session=provider,
    region_name=REGION
)

sm = session.client("secretsmanager")
response = sm.get_secret_value(SecretId="demo-document-bucket")  # noprod

creds = json.loads(response["SecretString"])
DB_HOST = creds["DB_HOST"]
DB_NAME = creds["DB_NAME"]
DB_USER = creds["DB_USER"]
DB_PASSWORD = creds["DB_PASSWORD"]
DB_PORT = creds["DB_PORT"]


# ============================================================
# 1) CONFIG
# ============================================================
CATALOG = "asset_api_preprod_3_gold"
SCHEMA_1 = "asset_master_data_gold"
SCHEMA_2 = "asset_static_data_gold"

# UC tables that live in schema 2
UC_SCHEMA_2_TABLES = {"plant_wapp", "unit_wapp"}

def uc_tbl(t: str) -> str:
    schema = SCHEMA_2 if t in UC_SCHEMA_2_TABLES else SCHEMA_1
    return f"{CATALOG}.{schema}.{t}"

# State = "curseur de synchro" (last delta version traitée) par table
STATE_BASE = "/Volumes/asset_api_webapp_dev/default/uc_state/uc_to_rds_state_noprod"

def state_path(t: str) -> str:
    return f"{STATE_BASE}/{t}.json"

# Postgres
PG_HOST = DB_HOST
PG_PORT = DB_PORT
PG_DB   = DB_NAME
PG_USER = DB_USER
PG_PWD  = DB_PASSWORD

jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"
jdbc_props = {"user": PG_USER, "password": PG_PWD, "driver": "org.postgresql.Driver"}

def pg_conn():
    return psycopg2.connect(host=PG_HOST, port=PG_PORT, database=PG_DB, user=PG_USER, password=PG_PWD)

# Behavior toggles
FULL_RECONCILE_DELETES_ON_SNAPSHOT = True
FALLBACK_TO_SNAPSHOT_ON_CDF_GAP = True

# Debug toggles
DEBUG_PG_ROW_BY_ROW_ON_ERROR = True     # fallback row-by-row to find first failing row
DEBUG_PG_ROW_BY_ROW_LIMIT = 5000        # guardrail
DEBUG_SQL_PREVIEW_CHARS = 300

# Optional: fail fast if ON CONFLICT column has no UNIQUE index/constraint
PRECHECK_UPSERT_CONSTRAINTS = False


# ============================================================
# 2) UTILS
# ============================================================
def pandas_none(x):
    return None if pd.isna(x) else x

def clean_source_id_str(colname: str):
    return F.regexp_replace(F.col(colname).cast("string"), " ", "").cast("string")

def load_last_version(table_name: str) -> int:
    try:
        txt = dbutils.fs.head(state_path(table_name), 10000)
        return int(json.loads(txt)["last_version"])
    except Exception:
        return -1

def save_last_version(table_name: str, last_version: int):
    dbutils.fs.mkdirs(STATE_BASE)
    payload = json.dumps({"last_version": int(last_version)})
    dbutils.fs.put(state_path(table_name), payload, overwrite=True)


def _merge_ctx(base: dict | None, extra: dict | None):
    d = {}
    if base:
        d.update(base)
    if extra:
        d.update(extra)
    return d


def table_exists_uc(full_name: str) -> bool:
    try:
        return spark.catalog.tableExists(full_name)
    except Exception:
        try:
            spark.table(full_name).limit(1).count()
            return True
        except Exception:
            return False

def latest_delta_version(uc_table: str) -> int:
    h = spark.sql(f"DESCRIBE HISTORY {uc_table} LIMIT 1").collect()
    return int(h[0]["version"])

def uc_object_type(full_name: str) -> str:
    try:
        t = spark.catalog.getTable(full_name)
        return "view" if str(t.tableType).upper() == "VIEW" else "table"
    except Exception:
        return "unknown"

def is_delta_table(full_name: str) -> bool:
    try:
        d = spark.sql(f"DESCRIBE DETAIL {full_name}").collect()[0].asDict()
        return str(d.get("format", "")).lower() == "delta"
    except Exception:
        return False

def latest_uc_version_or_fallback(full_name: str) -> int:
    if is_delta_table(full_name):
        return latest_delta_version(full_name)
    return -999


# ============================================================
# 3) CDF
# ============================================================
def read_cdf_batch(uc_table: str, start_version: int, end_version: int):
    if start_version > end_version:
        return None

    if not is_delta_table(uc_table):
        return None

    try:
        return (spark.read.format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", start_version)
                .option("endingVersion", end_version)
                .table(uc_table))
    except Exception as e:
        msg = str(e)
        if "DELTA_MISSING_CHANGE_DATA" in msg or "Missing change data" in msg:
            if FALLBACK_TO_SNAPSHOT_ON_CDF_GAP:
                log("CDF_GAP_FALLBACK_SNAPSHOT", uc_table=uc_table, start=start_version, end=end_version, err=msg)
                return None
        raise


# ============================================================
# 4) POSTGRES HELPERS + BETTER ERRORS
# ============================================================
def _pg_exc_details(e: Exception):
    diag = getattr(e, "diag", None)
    return {
        "err_type": type(e).__name__,
        "err_msg": str(e),
        "pgcode": getattr(e, "pgcode", None),
        "pgerror": getattr(e, "pgerror", None),
        "diag_sqlstate": getattr(diag, "sqlstate", None) if diag else None,
        "diag_message_primary": getattr(diag, "message_primary", None) if diag else None,
        "diag_constraint_name": getattr(diag, "constraint_name", None) if diag else None,
        "diag_table_name": getattr(diag, "table_name", None) if diag else None,
        "diag_column_name": getattr(diag, "column_name", None) if diag else None,
    }

def pg_table_columns(pg_table: str):
    conn = pg_conn()
    cur = conn.cursor()
    try:
        cur.execute("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema='public' AND table_name = %s
        """, (pg_table,))
        return set([r[0] for r in cur.fetchall()])
    finally:
        cur.close()
        conn.close()

NOT_NULL_STRING_DEFAULTS = {
    "global_technology_information": {"technology": "null"},
    "global_finance": {
        "smart_package_code": "null",
        "finance_entity_code": "null",
    },
}

def apply_not_null_default(pg_table: str, pg_col: str, col_expr):
    """
    Replace NULL/empty string with a configured default for PG NOT NULL string columns.
    """
    default = NOT_NULL_STRING_DEFAULTS.get(pg_table, {}).get(pg_col)
    if default is None:
        return col_expr
    c = col_expr.cast("string")
    return F.when(c.isNull() | (F.trim(c) == ""), F.lit(default)).otherwise(c)

def build_mapped_exprs(base_df, pg_table: str, mapping: dict, fill_missing: dict | None):
    """
    Build list of Spark expressions for mapped columns, applying:
    - fill_missing if UC col absent
    - NOT_NULL defaults if configured
    """
    cols_uc = set(base_df.columns)
    exprs = []

    for pg_c, uc_c in mapping.items():
        if uc_c in cols_uc:
            c = F.col(uc_c).cast("string")
            c = apply_not_null_default(pg_table, pg_c, c)
            exprs.append(c.alias(pg_c))
        else:
            if fill_missing and pg_c in fill_missing:
                c = fill_missing[pg_c]
                c = apply_not_null_default(pg_table, pg_c, c)
                exprs.append(c.alias(pg_c))
            else:
                default = NOT_NULL_STRING_DEFAULTS.get(pg_table, {}).get(pg_c)
                if default is not None:
                    exprs.append(F.lit(default).cast("string").alias(pg_c))
                else:
                    exprs.append(F.lit(None).cast("string").alias(pg_c))

    return exprs


def pg_fetchall(sql: str, params=None):
    conn = pg_conn()
    cur = conn.cursor()
    try:
        cur.execute(sql, params or ())
        return cur.fetchall()
    finally:
        cur.close()
        conn.close()

def pg_execute_many(sql: str, rows: list, page_size=1000, context=None):
    if not rows:
        return

    ctx = context or {}

    conn = pg_conn()
    cur = conn.cursor()
    try:
        log("PG_EXECUTE_MANY_START", rows=len(rows), page_size=page_size, **ctx)
        execute_batch(cur, sql, rows, page_size=page_size)
        conn.commit()
        log("PG_EXECUTE_MANY_OK", rows=len(rows), **ctx)

    except Exception as e:
        conn.rollback()

        # log enrichi (diag postgres si dispo)
        diag = getattr(e, "diag", None)
        log(
            "PG_EXECUTE_MANY_ERROR",
            err_type=type(e).__name__,
            err=str(e),
            diag_message_primary=getattr(diag, "message_primary", None) if diag else None,
            diag_constraint_name=getattr(diag, "constraint_name", None) if diag else None,
            diag_column_name=getattr(diag, "column_name", None) if diag else None,
            diag_table_name=getattr(diag, "table_name", None) if diag else None,
            **ctx
        )
        raise
    finally:
        cur.close()
        conn.close()

def pg_execute(sql: str, params=None, context: dict=None):
    context = context or {}
    conn = pg_conn()
    cur = conn.cursor()
    t0 = time.time()

    try:
        log("PG_EXECUTE_START", **context)
        cur.execute(sql, params or ())
        conn.commit()
        log("PG_EXECUTE_OK", elapsed_s=round(time.time()-t0, 3), **context)

    except Exception as e:
        conn.rollback()
        det = _pg_exc_details(e)
        log("PG_EXECUTE_ERROR",
            elapsed_s=round(time.time()-t0, 3),
            sql_preview=sql.strip().replace("\n"," ")[:DEBUG_SQL_PREVIEW_CHARS],
            **det, **context
        )
        raise

    finally:
        cur.close()
        conn.close()


# ============================================================
# 5) UPSERT / DELETE HELPERS (WITH LOGGING)
# ============================================================
def pg_has_unique_on(pg_table: str, col: str) -> bool:
    sql = """
    SELECT EXISTS (
      SELECT 1
      FROM pg_index i
      JOIN pg_class t ON t.oid = i.indrelid
      JOIN pg_attribute a ON a.attrelid = t.oid
      WHERE t.relname = %s
        AND i.indisunique
        AND a.attnum = ANY(i.indkey)
        AND a.attname = %s
    );
    """
    return bool(pg_fetchall(sql, (pg_table, col))[0][0])

def soft_delete_supported(pg_table: str) -> bool:
    cols = pg_table_columns(pg_table)
    return ("is_deleted" in cols) and ("deleted_at" in cols)

def soft_delete(pg_table: str, source_ids: list, context=None):
    if not source_ids:
        return 0

    ctx = _merge_ctx({"pg_table": pg_table}, context)

    if not soft_delete_supported(pg_table):
        log("SOFT_DELETE_SKIPPED_NO_COLS", count=len(source_ids), **ctx)
        return 0

    rows = [(sid,) for sid in source_ids if sid is not None]
    sql = f"UPDATE {pg_table} SET is_deleted=true, deleted_at=now() WHERE source_id=%s"

    log("SOFT_DELETE_START", count=len(rows), **ctx)
    pg_execute_many(sql, rows, context={**ctx, "action": "soft_delete"})
    log("SOFT_DELETE_OK", count=len(rows), **ctx)
    return len(rows)

def upsert_rows(pg_table: str, cols: list, rows: list, conflict_col="source_id", reactivate=True, context=None, exclude_update_cols=None):
    """
    Generic UPSERT:
    - INSERT id (uuid) + fields
    - ON CONFLICT(source_id) DO UPDATE SET ...
    - Optionally reactivate if soft-delete columns exist
    """
    if not rows:
        return 0

    ctx = _merge_ctx({"pg_table": pg_table}, context)

    cols_sql = ", ".join(cols)
    placeholders = ", ".join(["%s"] * len(cols))

    exclude_update_cols = set(exclude_update_cols or [])
    upd_cols = [c for c in cols if c not in ["id", conflict_col] and c not in exclude_update_cols]
    set_clause = ", ".join([f"{c}=EXCLUDED.{c}" for c in upd_cols])

    if reactivate and soft_delete_supported(pg_table):
        set_clause = (set_clause + ", is_deleted=false, deleted_at=NULL") if set_clause else "is_deleted=false, deleted_at=NULL"

    sql = f"""
        INSERT INTO {pg_table} ({cols_sql})
        VALUES ({placeholders})
        ON CONFLICT ({conflict_col}) DO UPDATE SET {set_clause};
    """

    # ✅ log sans doublons
    log("UPSERT_START", conflict_col=conflict_col, cols=len(cols), rows=len(rows), **ctx)

    pg_execute_many(sql, rows, context={**ctx, "action": "upsert", "conflict_col": conflict_col})

    log("UPSERT_OK", affected=len(rows), **ctx)
    return len(rows)


def upsert_rows_on_cols(pg_table: str, cols: list, rows: list, conflict_cols: list, reactivate=False, context=None):
    """
    UPSERT sur clé composite (ex: data_domain_id + version_id)
    """
    if not rows:
        return 0

    ctx = _merge_ctx({"pg_table": pg_table}, context)

    cols_sql = ", ".join(cols)
    placeholders = ", ".join(["%s"] * len(cols))
    conflict_sql = ", ".join(conflict_cols)

    upd_cols = [c for c in cols if c not in ["id"] + conflict_cols]
    set_clause = ", ".join([f"{c}=EXCLUDED.{c}" for c in upd_cols])

    # (reactivate inutile ici, ddv n'est pas soft-delete dans ton modèle)

    sql = f"""
        INSERT INTO {pg_table} ({cols_sql})
        VALUES ({placeholders})
        ON CONFLICT ({conflict_sql}) DO UPDATE SET {set_clause};
    """

    log("UPSERT_COMPOSITE_START", conflict_cols=conflict_cols, cols=len(cols), rows=len(rows), **ctx)
    pg_execute_many(sql, rows, context={**ctx, "action": "upsert_composite", "conflict_cols": conflict_cols})
    log("UPSERT_COMPOSITE_OK", affected=len(rows), **ctx)

    return len(rows)


# ============================================================
# 6) SNAPSHOT HELPERS
# ============================================================
def snapshot_df(uc_fullname: str):
    return spark.table(uc_fullname)

def snapshot_reconcile_deletes(pg_table: str, snapshot_source_ids: set, context=None):
    context = context or {}
    if not soft_delete_supported(pg_table):
        return 0
    existing = set([r[0] for r in pg_fetchall(f"SELECT source_id FROM {pg_table} WHERE is_deleted=false")])
    missing = list(existing - snapshot_source_ids)
    return soft_delete(pg_table, missing, context=context)


# ============================================================
# 7) PG MAPS (source_id VARCHAR -> uuid)
# ============================================================
def pg_map_source_to_uuid(pg_table: str, alias: str):
    df = spark.read.jdbc(url=jdbc_url, table=pg_table, properties=jdbc_props)
    return df.select(
        F.col("source_id").cast("string").alias(f"{alias}_source_id"),
        F.col("id").alias(f"{alias}_uuid")
    )

def pg_ddv_lookup():
    ddv = spark.read.jdbc(url=jdbc_url, table="data_domain_version", properties=jdbc_props).select(
        F.col("id").alias("ddv_uuid"),
        F.col("data_domain_id").alias("dd_uuid"),
        F.col("version_id").alias("v_uuid")
    )

    dd = spark.read.jdbc(url=jdbc_url, table="data_domain", properties=jdbc_props).select(
        F.col("id").alias("dd_uuid"),
        F.col("source_id").cast("string").alias("dd_source_id")
    )

    v = spark.read.jdbc(url=jdbc_url, table="version", properties=jdbc_props).select(
        F.col("id").alias("v_uuid"),
        F.col("source_id").cast("string").alias("v_source_id")
    )

    return (ddv.join(dd, "dd_uuid", "inner")
           .join(v, "v_uuid", "inner")
           .select("dd_source_id", "v_source_id", "ddv_uuid")
           .groupBy("dd_source_id", "v_source_id")
           .agg(F.min("ddv_uuid").alias("ddv_uuid")))


# ============================================================
# 8) PLANNER (detect changes)
# ============================================================
ALL_UC_TABLES = [
    "country",
    "data_domain",
    "gbu",
    "global_finance",
    "global_technology",
    "scenario",
    "signal",
    "signal_life_time",
    "unit_status",
    "version",
    "plant_wapp",
    "unit_wapp",
    "version_scenario_life_time",
]

def tables_with_changes():
    changed = []
    for t in ALL_UC_TABLES:
        uc = uc_tbl(t)

        if not table_exists_uc(uc):
            log("UC_TABLE_MISSING_SKIP", uc_table=uc)
            continue

        last = load_last_version(t)

        if not is_delta_table(uc):
            log("UC_NOT_DELTA_SNAPSHOT_MODE", uc_table=uc, object_type=uc_object_type(uc))
            changed.append(t)
            continue

        try:
            latest = latest_delta_version(uc)
        except Exception as e:
            log("UC_HISTORY_READ_FAIL", uc_table=uc, err=str(e))
            continue

        if latest > last:
            changed.append(t)

    return changed


# ============================================================
# 9) SYNC FUNCTIONS (your business logic, unchanged except context logs)
# ============================================================
def sync_dim_generic(uc_name: str, pg_name: str, mapping: dict, fill_missing: dict=None):
    uc = uc_tbl(uc_name)
    ctx = {"uc_table": uc, "uc_name": uc_name, "pg_table": pg_name}

    if not table_exists_uc(uc):
        log("UC_TABLE_MISSING", **ctx)
        return {"table": uc_name, "processed": False, "mode": "skip_missing", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    last = load_last_version(uc_name)
    latest = latest_uc_version_or_fallback(uc)
    log("SYNC_START", last=last, latest=latest, **ctx)

    # FIRST RUN snapshot
    if last < 0:
        mode = "snapshot"
        base = snapshot_df(uc).withColumn("source_id", clean_source_id_str("id"))

        exprs = build_mapped_exprs(base, pg_name, mapping, fill_missing)


        df_final = base.withColumn("id", F.expr("uuid()")).select(
            F.col("id").alias("id"),
            *exprs,
            F.col("source_id").alias("source_id")
        )

        pdf = df_final.toPandas()
        cols = ["id"] + list(mapping.keys()) + ["source_id"]

        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            row = [pandas_none(r.id)]
            for c in mapping.keys():
                row.append(pandas_none(getattr(r, c)))
            row.append(sid)
            rows.append(tuple(row))

        upserts = upsert_rows(pg_name, cols, rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})
        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest if latest != -999 else 0)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": 0, "upserts": upserts, "deletes": deletes}

    # VIEW/non-delta => snapshot each run
    if latest == -999:
        mode = "snapshot_view"
        base = snapshot_df(uc).withColumn("source_id", clean_source_id_str("id"))
        exprs = build_mapped_exprs(base, pg_name, mapping, fill_missing)


        df_final = base.withColumn("id", F.expr("uuid()")).select(
            F.col("id").alias("id"),
            *exprs,
            F.col("source_id").alias("source_id")
        )

        pdf = df_final.toPandas()
        cols = ["id"] + list(mapping.keys()) + ["source_id"]

        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            row = [pandas_none(r.id)]
            for c in mapping.keys():
                row.append(pandas_none(getattr(r, c)))
            row.append(sid)
            rows.append(tuple(row))

        upserts = upsert_rows(pg_name, cols, rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})
        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, last + 1)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": 0, "upserts": upserts, "deletes": deletes}

    # Incremental CDF
    start = last + 1
    if start > latest:
        log("SYNC_NOOP", reason="start>latest", **ctx)
        return {"table": uc_name, "processed": False, "mode": "noop", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    mode = "cdf"
    df = read_cdf_batch(uc, start, latest)

    # fallback snapshot if CDF missing
    if df is None:
        mode = "snapshot_fallback"
        base = snapshot_df(uc).withColumn("source_id", clean_source_id_str("id"))
        exprs = build_mapped_exprs(base, pg_name, mapping, fill_missing)


        df_final = base.withColumn("id", F.expr("uuid()")).select(
            F.col("id").alias("id"),
            *exprs,
            F.col("source_id").alias("source_id")
        )

        pdf = df_final.toPandas()
        cols = ["id"] + list(mapping.keys()) + ["source_id"]

        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            row = [pandas_none(r.id)]
            for c in mapping.keys():
                row.append(pandas_none(getattr(r, c)))
            row.append(sid)
            rows.append(tuple(row))

        upserts = upsert_rows(pg_name, cols, rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})
        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": 0, "upserts": upserts, "deletes": deletes}

    # normal cdf path
    df = df.withColumn("source_id", clean_source_id_str("id"))

    del_ids = (df.filter(F.col("_change_type") == "delete")
                 .select("source_id").distinct()
                 .toPandas()["source_id"].dropna().astype(str).tolist())
    deletes = soft_delete(pg_name, del_ids, context={**ctx, "mode": mode})

    up = df.filter(F.col("_change_type").isin("insert", "update_postimage"))

    exprs = build_mapped_exprs(up, pg_name, mapping, fill_missing)

    df_final = up.withColumn("id", F.expr("uuid()")).select(
        F.col("id").alias("id"),
        *exprs,
        F.col("source_id").alias("source_id")
    )

    pdf = df_final.toPandas()
    cols = ["id"] + list(mapping.keys()) + ["source_id"]

    rows = []
    for r in pdf.itertuples(index=False):
        sid = None if pd.isna(r.source_id) else str(r.source_id)
        if sid is None:
            continue
        row = [pandas_none(r.id)]
        for c in mapping.keys():
            row.append(pandas_none(getattr(r, c)))
        row.append(sid)
        rows.append(tuple(row))

    upserts = upsert_rows(pg_name, cols, rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})
    save_last_version(uc_name, latest)
    log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, **ctx)
    return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": 0, "upserts": upserts, "deletes": deletes}


# ============================================================
# 10) YOUR OTHER SYNC FUNCTIONS
# (I keep them the same as you had, just adding context logs
#  where they call upsert_rows / soft_delete / pg_execute_many)
# ============================================================

def sync_plant():
    uc_name = "plant_wapp"
    pg_name = "plant"
    uc = uc_tbl(uc_name)
    ctx = {"uc_table": uc, "uc_name": uc_name, "pg_table": pg_name}

    if not table_exists_uc(uc):
        log("UC_TABLE_MISSING", **ctx)
        return {"table": uc_name, "processed": False, "mode": "skip_missing", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    last = load_last_version(uc_name)
    latest = latest_uc_version_or_fallback(uc)
    log("SYNC_START", last=last, latest=latest, **ctx)

    def transform(base_df):
        base_df = base_df.withColumn("source_id", clean_source_id_str("id"))
        base_df = base_df.withColumn("country_source_id", clean_source_id_str("country_id"))

        country_map = pg_map_source_to_uuid("country", "country")

        total = base_df.count()
        joined = (base_df.join(country_map, base_df["country_source_id"] == country_map["country_source_id"], "inner")
                        .withColumn("id", F.expr("uuid()"))
                        .select(
                            F.col("id").alias("id"),
                            F.col("code").cast("string").alias("code"),
                            F.col("description").cast("string").alias("description"),
                            F.col("country_uuid").alias("country_id"),
                            F.col("source_id").alias("source_id"),
                        ))
        resolved = joined.count()
        skipped_fk = max(0, total - resolved)
        return joined, skipped_fk

    # snapshot first run
    if last < 0:
        mode = "snapshot"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((pandas_none(r.id), pandas_none(r.code), pandas_none(r.description), pandas_none(r.country_id), sid))

        upserts = upsert_rows(pg_name, ["id","code","description","country_id","source_id"], rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})
        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest if latest != -999 else 0)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    # incremental
    start = last + 1
    if latest == -999:
        # non-delta/view => snapshot each run
        mode = "snapshot_view"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)
        pdf = joined.toPandas()

        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((pandas_none(r.id), pandas_none(r.code), pandas_none(r.description), pandas_none(r.country_id), sid))

        upserts = upsert_rows(pg_name, ["id","code","description","country_id","source_id"], rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})
        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})
        save_last_version(uc_name, last + 1)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    if start > latest:
        log("SYNC_NOOP", reason="start>latest", **ctx)
        return {"table": uc_name, "processed": False, "mode": "noop", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    cdf = read_cdf_batch(uc, start, latest)
    if cdf is None:
        mode = "snapshot_fallback"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((pandas_none(r.id), pandas_none(r.code), pandas_none(r.description), pandas_none(r.country_id), sid))

        upserts = upsert_rows(pg_name, ["id","code","description","country_id","source_id"], rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})
        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})
        save_last_version(uc_name, latest)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    mode = "cdf"
    cdf = cdf.withColumn("source_id", clean_source_id_str("id"))

    del_ids = (cdf.filter(F.col("_change_type") == "delete")
                 .select("source_id").distinct()
                 .toPandas()["source_id"].dropna().astype(str).tolist())
    deletes = soft_delete(pg_name, del_ids, context={**ctx, "mode": mode})

    up = cdf.filter(F.col("_change_type").isin("insert","update_postimage"))
    joined, skipped_fk = transform(up)

    pdf = joined.toPandas()
    rows = []
    for r in pdf.itertuples(index=False):
        sid = None if pd.isna(r.source_id) else str(r.source_id)
        if sid is None:
            continue
        rows.append((pandas_none(r.id), pandas_none(r.code), pandas_none(r.description), pandas_none(r.country_id), sid))

    upserts = upsert_rows(pg_name, ["id","code","description","country_id","source_id"], rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})
    save_last_version(uc_name, latest)
    log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
    return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}


def sync_unit():
    uc_name = "unit_wapp"
    pg_name = "unit"
    uc = uc_tbl(uc_name)
    ctx = {"uc_table": uc, "uc_name": uc_name, "pg_table": pg_name}

    if not table_exists_uc(uc):
        log("UC_TABLE_MISSING", **ctx)
        return {"table": uc_name, "processed": False, "mode": "skip_missing", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    last = load_last_version(uc_name)
    latest = latest_uc_version_or_fallback(uc)
    log("SYNC_START", last=last, latest=latest, **ctx)

    def normalize_cols(df):
        cols = set(df.columns)
        if "gbu_id" in cols and "gb_information_id" not in cols:
            df = df.withColumnRenamed("gbu_id", "gb_information_id")
        if "global_technology_id" in cols and "global_technology_information_id" not in cols:
            df = df.withColumnRenamed("global_technology_id", "global_technology_information_id")
        if "finance_id" in cols and "global_finance_id" not in cols:
            df = df.withColumnRenamed("finance_id", "global_finance_id")
        return df

    def transform(base_df):
        base_df = normalize_cols(base_df)
        base_df = base_df.withColumn("source_id", clean_source_id_str("id"))

        base_df = (base_df
            .withColumn("plant_source_id", clean_source_id_str("plant_id"))
            .withColumn("unit_status_source_id", clean_source_id_str("unit_status_id"))
            .withColumn("gti_source_id", clean_source_id_str("global_technology_information_id") if "global_technology_information_id" in base_df.columns else F.lit(None).cast("string"))
            .withColumn("gbi_source_id", clean_source_id_str("gb_information_id") if "gb_information_id" in base_df.columns else F.lit(None).cast("string"))
            .withColumn("gf_source_id",  clean_source_id_str("global_finance_id") if "global_finance_id" in base_df.columns else F.lit(None).cast("string"))
        )

        plant_map = pg_map_source_to_uuid("plant", "plant")
        us_map    = pg_map_source_to_uuid("unit_status", "us")
        gti_map   = pg_map_source_to_uuid("global_technology_information", "gti")
        gbi_map   = pg_map_source_to_uuid("gb_information", "gbi")
        gf_map    = pg_map_source_to_uuid("global_finance", "gf")

        total = base_df.count()

        joined = (base_df
            .join(plant_map, base_df["plant_source_id"] == plant_map["plant_source_id"], "inner")
            .join(us_map,    base_df["unit_status_source_id"] == us_map["us_source_id"], "inner")
            .join(gti_map,   base_df["gti_source_id"] == gti_map["gti_source_id"], "left")
            .join(gbi_map,   base_df["gbi_source_id"] == gbi_map["gbi_source_id"], "left")
            .join(gf_map,    base_df["gf_source_id"]  == gf_map["gf_source_id"],  "left")
            .withColumn("id", F.expr("uuid()"))
            .select(
                F.col("id").alias("id"),
                F.col("code").cast("string").alias("code"),
                F.col("description").cast("string").alias("description"),
                F.col("plant_uuid").alias("plant_id"),
                F.col("us_uuid").alias("unit_status_id"),
                F.col("gti_uuid").alias("global_technology_information_id"),
                F.col("gbi_uuid").alias("gb_information_id"),
                F.col("gf_uuid").alias("global_finance_id"),
                F.col("source_id").alias("source_id"),
            )
        )

        resolved = joined.count()
        skipped_fk = max(0, total - resolved)
        return joined, skipped_fk

    # ---------- FIRST RUN: SNAPSHOT ----------
    if last < 0:
        mode = "snapshot"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.code),
                pandas_none(r.description),
                pandas_none(r.plant_id),
                pandas_none(r.unit_status_id),
                pandas_none(r.global_technology_information_id),
                pandas_none(r.gb_information_id),
                pandas_none(r.global_finance_id),
                sid
            ))

        upserts = upsert_rows(pg_name,
            ["id","code","description","plant_id","unit_status_id","global_technology_information_id","gb_information_id","global_finance_id","source_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest if latest != -999 else 0)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    # ---------- NON-DELTA (VIEW): SNAPSHOT EACH RUN ----------
    if latest == -999:
        mode = "snapshot_view"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.code),
                pandas_none(r.description),
                pandas_none(r.plant_id),
                pandas_none(r.unit_status_id),
                pandas_none(r.global_technology_information_id),
                pandas_none(r.gb_information_id),
                pandas_none(r.global_finance_id),
                sid
            ))

        upserts = upsert_rows(pg_name,
            ["id","code","description","plant_id","unit_status_id","global_technology_information_id","gb_information_id","global_finance_id","source_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, last + 1)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    # ---------- INCREMENTAL: CDF ----------
    start = last + 1
    if start > latest:
        log("SYNC_NOOP", reason="start>latest", **ctx)
        return {"table": uc_name, "processed": False, "mode": "noop", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    cdf = read_cdf_batch(uc, start, latest)

    # fallback snapshot if CDF missing
    if cdf is None:
        mode = "snapshot_fallback"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.code),
                pandas_none(r.description),
                pandas_none(r.plant_id),
                pandas_none(r.unit_status_id),
                pandas_none(r.global_technology_information_id),
                pandas_none(r.gb_information_id),
                pandas_none(r.global_finance_id),
                sid
            ))

        upserts = upsert_rows(pg_name,
            ["id","code","description","plant_id","unit_status_id","global_technology_information_id","gb_information_id","global_finance_id","source_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    # normal CDF path
    mode = "cdf"
    cdf = normalize_cols(cdf).withColumn("source_id", clean_source_id_str("id"))

    del_ids = (cdf.filter(F.col("_change_type") == "delete")
                 .select("source_id").distinct()
                 .toPandas()["source_id"].dropna().astype(str).tolist())
    deletes = soft_delete(pg_name, del_ids, context={**ctx, "mode": mode})

    up = cdf.filter(F.col("_change_type").isin("insert","update_postimage"))
    joined, skipped_fk = transform(up)

    pdf = joined.toPandas()
    rows = []
    for r in pdf.itertuples(index=False):
        sid = None if pd.isna(r.source_id) else str(r.source_id)
        if sid is None:
            continue
        rows.append((
            pandas_none(r.id),
            pandas_none(r.code),
            pandas_none(r.description),
            pandas_none(r.plant_id),
            pandas_none(r.unit_status_id),
            pandas_none(r.global_technology_information_id),
            pandas_none(r.gb_information_id),
            pandas_none(r.global_finance_id),
            sid
        ))

    upserts = upsert_rows(pg_name,
        ["id","code","description","plant_id","unit_status_id","global_technology_information_id","gb_information_id","global_finance_id","source_id"],
        rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})

    save_last_version(uc_name, latest)
    log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
    return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}


def parse_month_any(colname: str):
    c = F.trim(F.col(colname).cast("string"))

    # normalise FR -> EN (mois abrégés)
    c_norm = F.lower(c)
    c_norm = F.regexp_replace(c_norm, r"janv", "jan")
    c_norm = F.regexp_replace(c_norm, r"févr|fevr", "feb")
    c_norm = F.regexp_replace(c_norm, r"mars", "mar")
    c_norm = F.regexp_replace(c_norm, r"avr", "apr")
    c_norm = F.regexp_replace(c_norm, r"mai", "may")
    c_norm = F.regexp_replace(c_norm, r"juin", "jun")
    c_norm = F.regexp_replace(c_norm, r"juil", "jul")
    c_norm = F.regexp_replace(c_norm, r"août|aout", "aug")
    c_norm = F.regexp_replace(c_norm, r"sept", "sep")
    c_norm = F.regexp_replace(c_norm, r"déc|dec", "dec")

    return F.coalesce(
        F.try_to_timestamp(F.concat(F.lit("01/"), c_norm), F.lit("dd/MM/yyyy")),
        F.try_to_timestamp(F.concat(c_norm, F.lit("-01")), F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(F.concat(F.lit("01/"), c_norm), F.lit("dd/MMM-yy")),
        F.try_to_timestamp(F.concat(F.lit("01/"), c_norm), F.lit("dd/MMM-yyyy")),
        F.try_to_timestamp(c_norm)
    )
def sync_data_domain_version_from_exercise():
    """
    Derive ddv pairs from UC exercise snapshot/CDF
    Insert missing pairs into RDS data_domain_version
    - forbidden_unit_status = '{}' (jsonb)
    - no delete
    """
    uc_name = "version_scenario_life_time"
    state_name = "data_domain_version__derived"
    uc = uc_tbl(uc_name)

    ctx = {"uc_table": uc, "uc_name": uc_name, "pg_table": "data_domain_version", "state_name": state_name}

    if not table_exists_uc(uc):
        log("UC_TABLE_MISSING_SKIP_DDV", **ctx)
        return {"table": state_name, "processed": False, "mode": "skip_missing", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    last = load_last_version(state_name)
    latest = latest_uc_version_or_fallback(uc)
    log("DDV_DERIVE_START", last=last, latest=latest, **ctx)

    def pairs_from_df(df):
        return (df
            .withColumn("dd_source_id", clean_source_id_str("data_domain_id"))
            .withColumn("v_source_id",  clean_source_id_str("version_id"))
            .select("dd_source_id","v_source_id")
            .dropna()
            .dropDuplicates(["dd_source_id","v_source_id"])
        )

    # FIRST RUN snapshot
    if last < 0:
        mode = "snapshot"
        base = snapshot_df(uc)
        pairs = pairs_from_df(base)
    else:
        start = last + 1

        if latest == -999:
            mode = "snapshot_view"
            base = snapshot_df(uc)
            pairs = pairs_from_df(base)
        else:
            if start > latest:
                log("DDV_DERIVE_NOOP", reason="start>latest", **ctx)
                return {"table": state_name, "processed": False, "mode": "noop", "skipped_fk": 0, "upserts": 0, "deletes": 0}

            cdf = read_cdf_batch(uc, start, latest)
            if cdf is None:
                mode = "snapshot_fallback"
                base = snapshot_df(uc)
                pairs = pairs_from_df(base)
            else:
                mode = "cdf"
                up = cdf.filter(F.col("_change_type").isin("insert","update_postimage"))
                pairs = pairs_from_df(up)

    dd_map = pg_map_source_to_uuid("data_domain", "dd")
    v_map  = pg_map_source_to_uuid("version", "v")

    total = pairs.count()

    resolved = (pairs
        .join(dd_map, pairs["dd_source_id"] == dd_map["dd_source_id"], "inner")
        .join(v_map,  pairs["v_source_id"]  == v_map["v_source_id"],  "inner")
        .select(
            F.expr("uuid()").alias("id"),
            F.col("dd_uuid").alias("data_domain_id"),
            F.col("v_uuid").alias("version_id"),
        )
    )

    ok = resolved.count()
    skipped_fk = max(0, total - ok)

    pdf = resolved.toPandas()

    rows = []
    for r in pdf.itertuples(index=False):
        rows.append((
            pandas_none(r.id),
            pandas_none(r.data_domain_id),
            pandas_none(r.version_id),
            "{}"  # forbidden_unit_status
        ))

    # UPSERT sur (data_domain_id, version_id) => plus jamais de doublons
    upserts = upsert_rows_on_cols(
        "data_domain_version",
        cols=["id", "data_domain_id", "version_id", "forbidden_unit_status"],
        rows=rows,
        conflict_cols=["data_domain_id", "version_id"],
        reactivate=False,
        context={**ctx, "mode": mode}
    )

    log("DDV_DERIVE_OK", mode=mode, upserts=upserts, skipped_fk=skipped_fk, **ctx)

    # advance state
    if latest == -999:
        save_last_version(state_name, last + 1)
    else:
        save_last_version(state_name, latest)

    log("DDV_DERIVE_OK", mode=mode, inserted=len(rows), skipped_fk=skipped_fk, **ctx)
    return {"table": state_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": len(rows), "deletes": 0}

def sync_exercise():
    uc_name = "version_scenario_life_time"
    pg_name = "exercise"
    uc = uc_tbl(uc_name)
    ctx = {"uc_table": uc, "uc_name": uc_name, "pg_table": pg_name}

    if not table_exists_uc(uc):
        log("UC_TABLE_MISSING", **ctx)
        return {"table": uc_name, "processed": False, "mode": "skip_missing", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    last = load_last_version(uc_name)
    latest = latest_uc_version_or_fallback(uc)
    log("SYNC_START", last=last, latest=latest, **ctx)

    def transform(base_df):
        base_df = base_df.withColumn("source_id", clean_source_id_str("id"))
        base_df = (base_df
            .withColumn("scenario_source_id", clean_source_id_str("scenario_id"))
            .withColumn("dd_source_id", clean_source_id_str("data_domain_id"))
            .withColumn("v_source_id",  clean_source_id_str("version_id"))
            .withColumn("start_year", F.col("start_year").cast("int"))
            .withColumn("end_year",   F.col("end_year").cast("int"))
            .withColumn("month_start", parse_month_any("month_start"))
            .withColumn("month_end",   parse_month_any("month_end"))
            .withColumn("is_active", F.lit(False).cast("boolean"))
        )

        scenario_map = pg_map_source_to_uuid("scenario", "sc")
        ddv_lu = pg_ddv_lookup()

        total = base_df.count()

        joined = (base_df
            .join(scenario_map, base_df["scenario_source_id"] == scenario_map["sc_source_id"], "inner")
            .join(ddv_lu, on=["dd_source_id","v_source_id"], how="inner")
            .withColumn("id", F.expr("uuid()"))
            .select(
                F.col("id").alias("id"),
                F.col("code").cast("string").alias("code"),
                F.col("description").cast("string").alias("description"),
                F.col("start_year").alias("start_year"),
                F.col("end_year").alias("end_year"),
                F.col("month_start").alias("month_start"),
                F.col("month_end").alias("month_end"),
                F.col("is_active").alias("is_active"),
                F.col("sc_uuid").alias("scenario_id"),
                F.col("ddv_uuid").alias("data_domain_version_id"),
                F.col("source_id").alias("source_id"),
            )
        )

        resolved = joined.count()
        skipped_fk = max(0, total - resolved)
        return joined, skipped_fk

    # ---------- FIRST RUN: SNAPSHOT ----------
    if last < 0:
        mode = "snapshot"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.code),
                pandas_none(r.description),
                None if pd.isna(r.start_year) else int(r.start_year),
                None if pd.isna(r.end_year) else int(r.end_year),
                None if pd.isna(r.month_start) else r.month_start,
                None if pd.isna(r.month_end) else r.month_end,
                False,
                pandas_none(r.scenario_id),
                pandas_none(r.data_domain_version_id),
                sid
            ))

        upserts = upsert_rows(pg_name,
            ["id","code","description","start_year","end_year","month_start","month_end","is_active","scenario_id","data_domain_version_id","source_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode},
            exclude_update_cols=["is_active"])

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest if latest != -999 else 0)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    # ---------- NON-DELTA (VIEW): SNAPSHOT EACH RUN ----------
    if latest == -999:
        mode = "snapshot_view"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.code),
                pandas_none(r.description),
                None if pd.isna(r.start_year) else int(r.start_year),
                None if pd.isna(r.end_year) else int(r.end_year),
                None if pd.isna(r.month_start) else r.month_start,
                None if pd.isna(r.month_end) else r.month_end,
                False,
                pandas_none(r.scenario_id),
                pandas_none(r.data_domain_version_id),
                sid
            ))

        upserts = upsert_rows(pg_name,
            ["id","code","description","start_year","end_year","month_start","month_end","is_active","scenario_id","data_domain_version_id","source_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode},
            exclude_update_cols=["is_active"])

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, last + 1)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    # ---------- INCREMENTAL: CDF ----------
    start = last + 1
    if start > latest:
        log("SYNC_NOOP", reason="start>latest", **ctx)
        return {"table": uc_name, "processed": False, "mode": "noop", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    cdf = read_cdf_batch(uc, start, latest)

    if cdf is None:
        mode = "snapshot_fallback"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.code),
                pandas_none(r.description),
                None if pd.isna(r.start_year) else int(r.start_year),
                None if pd.isna(r.end_year) else int(r.end_year),
                None if pd.isna(r.month_start) else r.month_start,
                None if pd.isna(r.month_end) else r.month_end,
                False,
                pandas_none(r.scenario_id),
                pandas_none(r.data_domain_version_id),
                sid
            ))

        upserts = upsert_rows(pg_name,
            ["id","code","description","start_year","end_year","month_start","month_end","is_active","scenario_id","data_domain_version_id","source_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode},
            exclude_update_cols=["is_active"])

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    mode = "cdf"
    cdf = cdf.withColumn("source_id", clean_source_id_str("id"))

    del_ids = (cdf.filter(F.col("_change_type") == "delete")
                 .select("source_id").distinct()
                 .toPandas()["source_id"].dropna().astype(str).tolist())
    deletes = soft_delete(pg_name, del_ids, context={**ctx, "mode": mode})

    up = cdf.filter(F.col("_change_type").isin("insert","update_postimage"))
    joined, skipped_fk = transform(up)

    pdf = joined.toPandas()
    rows = []
    for r in pdf.itertuples(index=False):
        sid = None if pd.isna(r.source_id) else str(r.source_id)
        if sid is None:
            continue
        rows.append((
            pandas_none(r.id),
            pandas_none(r.code),
            pandas_none(r.description),
            None if pd.isna(r.start_year) else int(r.start_year),
            None if pd.isna(r.end_year) else int(r.end_year),
            None if pd.isna(r.month_start) else r.month_start,
            None if pd.isna(r.month_end) else r.month_end,
            False,
            pandas_none(r.scenario_id),
            pandas_none(r.data_domain_version_id),
            sid
        ))

    upserts = upsert_rows(pg_name,
            ["id","code","description","start_year","end_year","month_start","month_end","is_active","scenario_id","data_domain_version_id","source_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode},
            exclude_update_cols=["is_active"])

    save_last_version(uc_name, latest)
    log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
    return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

def sync_signal_life_time():
    uc_name = "signal_life_time"
    pg_name = "signal_life_time"
    uc = uc_tbl(uc_name)
    ctx = {"uc_table": uc, "uc_name": uc_name, "pg_table": pg_name}

    if not table_exists_uc(uc):
        log("UC_TABLE_MISSING", **ctx)
        return {"table": uc_name, "processed": False, "mode": "skip_missing", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    last = load_last_version(uc_name)
    latest = latest_uc_version_or_fallback(uc)
    log("SYNC_START", last=last, latest=latest, **ctx)

    def transform(base_df):
        base_df = base_df.withColumn("source_id", clean_source_id_str("id"))

        cols = set(base_df.columns)
        lower_expr = F.col("lower_bound").cast("double") if "lower_bound" in cols else F.lit(None).cast("double")
        upper_expr = F.col("upper_bound").cast("double") if "upper_bound" in cols else F.lit(None).cast("double")

        base_df = (base_df
            .withColumn("unit_source_id", clean_source_id_str("unit_id"))
            .withColumn("signal_source_id", clean_source_id_str("signal_id"))
            .withColumn("dd_source_id", clean_source_id_str("data_domain_id"))
            .withColumn("v_source_id",  clean_source_id_str("version_id"))
            .withColumn("lower_bound", lower_expr)
            .withColumn("upper_bound", upper_expr)
        )

        unit_map   = pg_map_source_to_uuid("unit", "u")
        signal_map = pg_map_source_to_uuid("signal", "s")
        ddv_lu     = pg_ddv_lookup()

        total = base_df.count()

        joined = (base_df
            .join(unit_map,   base_df["unit_source_id"]   == unit_map["u_source_id"], "inner")
            .join(signal_map, base_df["signal_source_id"] == signal_map["s_source_id"], "inner")
            .join(ddv_lu, on=["dd_source_id","v_source_id"], how="inner")
            .withColumn("id", F.expr("uuid()"))
            .select(
                F.col("id").alias("id"),
                F.col("signal_granularity").cast("string").alias("signal_granularity"),
                F.col("lower_bound").alias("lower_bound"),
                F.col("upper_bound").alias("upper_bound"),
                F.col("source_id").alias("source_id"),
                F.col("u_uuid").alias("unit_id"),
                F.col("ddv_uuid").alias("data_domain_version_id"),
                F.col("s_uuid").alias("signal_id"),
            )
        )

        resolved = joined.count()
        skipped_fk = max(0, total - resolved)
        return joined, skipped_fk

    # FIRST RUN snapshot
    if last < 0:
        mode = "snapshot"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.signal_granularity),
                pandas_none(r.lower_bound),
                pandas_none(r.upper_bound),
                sid,
                pandas_none(r.unit_id),
                pandas_none(r.data_domain_version_id),
                pandas_none(r.signal_id),
            ))

        upserts = upsert_rows(pg_name,
            ["id","signal_granularity","lower_bound","upper_bound","source_id","unit_id","data_domain_version_id","signal_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest if latest != -999 else 0)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    # NON-DELTA snapshot each run
    if latest == -999:
        mode = "snapshot_view"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.signal_granularity),
                pandas_none(r.lower_bound),
                pandas_none(r.upper_bound),
                sid,
                pandas_none(r.unit_id),
                pandas_none(r.data_domain_version_id),
                pandas_none(r.signal_id),
            ))

        upserts = upsert_rows(pg_name,
            ["id","signal_granularity","lower_bound","upper_bound","source_id","unit_id","data_domain_version_id","signal_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, last + 1)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    # incremental CDF
    start = last + 1
    if start > latest:
        log("SYNC_NOOP", reason="start>latest", **ctx)
        return {"table": uc_name, "processed": False, "mode": "noop", "skipped_fk": 0, "upserts": 0, "deletes": 0}

    cdf = read_cdf_batch(uc, start, latest)
    if cdf is None:
        mode = "snapshot_fallback"
        base = snapshot_df(uc)
        joined, skipped_fk = transform(base)

        pdf = joined.toPandas()
        rows = []
        snapshot_ids = set()
        for r in pdf.itertuples(index=False):
            sid = None if pd.isna(r.source_id) else str(r.source_id)
            if sid is None:
                continue
            snapshot_ids.add(sid)
            rows.append((
                pandas_none(r.id),
                pandas_none(r.signal_granularity),
                pandas_none(r.lower_bound),
                pandas_none(r.upper_bound),
                sid,
                pandas_none(r.unit_id),
                pandas_none(r.data_domain_version_id),
                pandas_none(r.signal_id),
            ))

        upserts = upsert_rows(pg_name,
            ["id","signal_granularity","lower_bound","upper_bound","source_id","unit_id","data_domain_version_id","signal_id"],
            rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})

        deletes = 0
        if FULL_RECONCILE_DELETES_ON_SNAPSHOT:
            deletes = snapshot_reconcile_deletes(pg_name, snapshot_ids, context={**ctx, "mode": mode})

        save_last_version(uc_name, latest)
        log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
        return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}

    mode = "cdf"
    cdf = cdf.withColumn("source_id", clean_source_id_str("id"))

    del_ids = (cdf.filter(F.col("_change_type") == "delete")
                 .select("source_id").distinct()
                 .toPandas()["source_id"].dropna().astype(str).tolist())
    deletes = soft_delete(pg_name, del_ids, context={**ctx, "mode": mode})

    up = cdf.filter(F.col("_change_type").isin("insert","update_postimage"))
    joined, skipped_fk = transform(up)

    pdf = joined.toPandas()
    rows = []
    for r in pdf.itertuples(index=False):
        sid = None if pd.isna(r.source_id) else str(r.source_id)
        if sid is None:
            continue
        rows.append((
            pandas_none(r.id),
            pandas_none(r.signal_granularity),
            pandas_none(r.lower_bound),
            pandas_none(r.upper_bound),
            sid,
            pandas_none(r.unit_id),
            pandas_none(r.data_domain_version_id),
            pandas_none(r.signal_id),
        ))

    upserts = upsert_rows(pg_name,
        ["id","signal_granularity","lower_bound","upper_bound","source_id","unit_id","data_domain_version_id","signal_id"],
        rows, conflict_col="source_id", reactivate=True, context={**ctx, "mode": mode})

    save_last_version(uc_name, latest)
    log("SYNC_OK", mode=mode, upserts=upserts, deletes=deletes, skipped_fk=skipped_fk, **ctx)
    return {"table": uc_name, "processed": True, "mode": mode, "skipped_fk": skipped_fk, "upserts": upserts, "deletes": deletes}



# ============================================================
# 11) ROUTING + ORDER
# ============================================================
DIM_SPECS = {
    "country": ("country", {"code": "code", "description": "description"}, None),
    "data_domain": ("data_domain", {"code": "code", "description": "description"}, None),
    "gbu": ("gb_information", {"gb_name": "gb_name", "gbu_name": "gbu_name"}, None),
    "global_finance": (
    "global_finance",
    {
        "smart_package_code": "smart_package_code",
        "smart_package_label": "smart_package_label",
        "finance_entity_code": "smart_entity_code",
        "finance_entity_label": "smart_entity_label",
    },
    {
        # si la colonne UC est absente -> fallback, mais ici elle existe; le problème est surtout NULL/empty.
        # On garde quand même une valeur "null" safe si UC ne la fournit pas du tout.
        "finance_entity_code": F.lit("null").cast("string"),
    }
),

    "global_technology": ("global_technology_information", {
        "technology": "main_technology",
        "sub_technology": "sub_technology",
        "technology_type": "technology_class",
    }, {
        "technology": F.lit("null").cast("string"),
        "sub_technology": F.lit(None).cast("string"),
        "technology_type": F.lit(None).cast("string"),
    }),
    "version": ("version", {"code": "code", "description": "description"}, None),
    "scenario": ("scenario", {"code": "code", "description": "description"}, None),
    "unit_status": ("unit_status", {"code": "code", "description": "description"}, None),
    "signal": ("signal", {
        "code": "code",
        "description": "description",
        "type": "type",
        "unit_of_measurement_code": "unit_of_measurement_code",
        "unit_of_measurement_description": "unit_of_measurement_description",
    }, {"type": F.lit(None).cast("string")}),
}

ORDER = [
    "country",
    "data_domain",
    "gbu",
    "global_technology",
    "global_finance",
    "version",
    "scenario",
    "unit_status",
    "signal",
    "plant_wapp",
    "unit_wapp",
    "version_scenario_life_time",
    "data_domain_version__derived",
    "signal_life_time",
]

def run_one_table(name: str):
    # --- DIM tables via generic sync ---
    if name in DIM_SPECS:
        pg_name, mapping, fill = DIM_SPECS[name]
        return sync_dim_generic(name, pg_name, mapping, fill_missing=fill)

    # --- Specialized tables (FK / derived logic) ---
    if name == "plant_wapp":
        return sync_plant()

    if name == "unit_wapp":
        return sync_unit()

    if name == "version_scenario_life_time":
        return sync_exercise()

    if name == "data_domain_version__derived":
        return sync_data_domain_version_from_exercise()

    if name == "signal_life_time":
        return sync_signal_life_time()

    # --- Unknown ---
    log("UNKNOWN_TABLE", table=name)
    return {"table": name, "processed": False, "mode": "unknown", "skipped_fk": 0, "upserts": 0, "deletes": 0}


# ============================================================
# 12) UC PRECHECK
# ============================================================
missing_uc_tables = []
for t in ALL_UC_TABLES:
    full = uc_tbl(t)
    if not table_exists_uc(full):
        missing_uc_tables.append(full)

if missing_uc_tables:
    msg = "❌ Pipeline aborted: missing UC tables:\n" + "\n".join([f"  - {t}" for t in missing_uc_tables])
    raise Exception(msg)

log("UC_PRECHECK_OK", count=len(ALL_UC_TABLES))


# ============================================================
# 13) MAIN
# ============================================================
MAX_PASSES = 3

dbutils.widgets.text("FORCE_TABLES", "")
FORCE_TABLES_RAW = dbutils.widgets.get("FORCE_TABLES").strip()
FORCE_SET = {x.strip() for x in FORCE_TABLES_RAW.split(",") if x.strip()}

if FORCE_SET:
    changed_tables = list(FORCE_SET)
else:
    changed_tables = tables_with_changes()

if FORCE_SET:
    pending = [t for t in ORDER if t in FORCE_SET]
else:
    pending = [t for t in ORDER if t in changed_tables or (t == "data_domain_version__derived" and "version_scenario_life_time" in changed_tables)]

if not pending and not FORCE_SET:
    log("NOTHING_TO_DO", reason="no changed tables")
else:
    all_stats = []
    to_retry = pending[:]

    for p in range(1, MAX_PASSES + 1):
        log("PASS_START", pass_n=p, max_passes=MAX_PASSES, pending=to_retry)
        retry_next = []
        pass_stats = []

        for t in ORDER:
            if FORCE_SET:
                to_run = (t in FORCE_SET)
            else:
                to_run = (t in to_retry)

            if not to_run:
                continue

            log("TABLE_RUN_START", table=t, pass_n=p, forced=bool(FORCE_SET))

            try:
                stats = run_one_table(t)
                pass_stats.append(stats)
                log("TABLE_RUN_OK", table=t, pass_n=p, stats=stats)

                if stats.get("skipped_fk", 0) > 0 and t in ["plant_wapp","unit_wapp","data_domain_version__derived","version_scenario_life_time","signal_life_time"]:
                    retry_next.append(t)

            except Exception as e:
                log("TABLE_RUN_FAIL", table=t, pass_n=p, err_type=type(e).__name__, err=str(e))
                # prints python stack too
                print(traceback.format_exc())
                raise

        all_stats.extend(pass_stats)

        if not retry_next:
            log("PASS_STOP_EARLY", pass_n=p, reason="no FK-skips remaining")
            break

        to_retry = list(dict.fromkeys(retry_next))
        log("PASS_END", pass_n=p, retry_next=to_retry)

    log("SUMMARY_START", rows=len(all_stats))
    if all_stats:
        df_stats = pd.DataFrame(all_stats)
        display(spark.createDataFrame(df_stats))
    log("DONE")


# ============================================================
# 15) FINALIZE: REFRESH MATERIALIZED VIEWS
# ============================================================
log("MV_REFRESH_PHASE_START")

pg_execute("REFRESH MATERIALIZED VIEW mv_data_domain_signals;", context={"action": "refresh_mv", "mv": "mv_data_domain_signals"})
pg_execute("REFRESH MATERIALIZED VIEW mv_unit_signals_by_version;", context={"action": "refresh_mv", "mv": "mv_unit_signals_by_version"})

log("MV_REFRESH_PHASE_OK", views=["mv_data_domain_signals", "mv_unit_signals_by_version"])
